# Encoding Functional Drift — SLURM Orchestrator

Tests whether Poisson GLM encoding models trained on one day generalize to future days,
with permutation testing for significance.

**Phase 1** (one SLURM job per (patient, train_date)):  
Fit a full-day model on all 9 AM–11 PM words using alpha hyperparameters from the existing
nested-CV per-day results. Saves model, preprocessing bundle, and training word indices.

**Phase 2** (one SLURM job per (patient, train_date), depends on Phase 1):  
For each subsequent test_date, apply the Phase-1 bundle + model to test-day words and run
N permutation tests (shuffle training X, refit bundle+GLM, evaluate) in parallel via joblib.

Set `SPEECH_TYPE` to `'filtered'` or `'patient'` to select the source model run.

In [1]:
from pathlib import Path
import subprocess

import dill as pickle
import numpy as np
import pandas as pd

In [10]:
# ── Paths ──────────────────────────────────────────────────────────────────────
PROJ_DIR  = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247')
VAD_ROOT  = PROJ_DIR / 'vad_new'

WORKER_TRAIN = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                    '/functional_drift/encoding_drift_train_worker.py')
WORKER_TEST  = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                    '/functional_drift/encoding_drift_test_worker.py')
PYTHON_BIN   = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'

PATIENTS       = None   # None => scan all Y* folders
FORCE_RESUBMIT = True

# ── Speech type ────────────────────────────────────────────────────────────────
SPEECH_TYPE = 'filtered'   # 'filtered' or 'patient'

SOURCE_RUN = {
    'filtered': 'word_level_duration_cv_filtered_speech_per_day',
    'patient':  'word_level_duration_cv_patient_speech_per_day',
}[SPEECH_TYPE]

DRIFT_SUBDIR = f'encoding_{SPEECH_TYPE}'

# ── Model params (must match the source run) ──────────────────────────────────
SPIKE_OFFSET_IDX = 0
GPT2_LAYER       = -1
N_PCA            = 100

# ── Permutation testing ────────────────────────────────────────────────────────
N_PERMUTATIONS = 50   # permutations per (train_date, test_date)
N_JOBS_PERM    = 4     # joblib workers within Phase-2 job

# ── SLURM — Phase 1 (GPU, fits full-day GLM) ──────────────────────────────────
CONDA_ACTIVATE = 'source /scratch/tahaismail424/.bash_profile && conda activate speech_247_env'

P1_PARTITION = 'Universal'
P1_GRES      = 'gpu:1'
P1_CPUS      = 8
P1_MEM       = '64G'
P1_TIME      = '08:00:00'

# ── SLURM — Phase 2 (CPU, permutation testing) ────────────────────────────────
P2_PARTITION = 'Universal'
P2_GRES      = ''
P2_CPUS      = N_JOBS_PERM * 2
P2_MEM       = '32G'
P2_TIME      = '24:00:00'

# ── Logging ───────────────────────────────────────────────────────────────────
LOGS_DIR    = VAD_ROOT / f'encoding_drift_{SPEECH_TYPE}_slurm_logs'
SCRIPTS_DIR = VAD_ROOT / f'encoding_drift_{SPEECH_TYPE}_slurm_scripts'
LOGS_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print('vad root:   ', VAD_ROOT)
print('source run: ', SOURCE_RUN)
print('drift dir:  ', DRIFT_SUBDIR)
print(f'permutations: {N_PERMUTATIONS} x n_jobs={N_JOBS_PERM}')

vad root:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new
source run:  word_level_duration_cv_filtered_speech_per_day
drift dir:   encoding_filtered
permutations: 50 x n_jobs=4


## 1. Discover patients and training days

In [7]:
def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def resolve_patient_inputs(patient):
    r = VAD_ROOT / patient
    emb = first_existing([r / 'embeddings' / f'{patient}_gpt2_embeddings.npy',
                           r / 'all_convo_recording' / 'all_words_filtered_all_layers_gpt2.npy'])
    cnt = first_existing([r / 'neural_embeddings' / 'word_spike_counts_offsets_all.npy',
                           r / 'all_convo_recording' / 'word_spike_counts_offsets_all.npy'])
    dur = first_existing([r / 'neural_embeddings' / 'word_durs.npy',
                           r / 'all_convo_recording' / 'word_durs.npy'])
    return emb, cnt, dur


def get_model_days(patient):
    """Days with existing nested-CV results (have cv pkl + tar)."""
    base = VAD_ROOT / patient / 'encoding' / SOURCE_RUN
    if not base.exists():
        return []
    days = []
    for d in sorted(base.iterdir()):
        if (d.is_dir()
                and (d / f'{patient}_encoding_results_cv.pkl').exists()
                and (d / f'{patient}_encoding_models_cv.tar').exists()):
            idx_files = list(d.glob(f'{patient}_*_word_idx.npy'))
            if idx_files:
                days.append((d.name, d / f'{patient}_encoding_results_cv.pkl', idx_files[0]))
    return days


def get_test_days(patient):
    """Days with word_idx files (can serve as test days)."""
    base = VAD_ROOT / patient / 'encoding' / SOURCE_RUN
    if not base.exists():
        return []
    return [d.name for d in sorted(base.iterdir())
            if d.is_dir() and list(d.glob(f'{patient}_*_word_idx.npy'))]


if PATIENTS is None:
    all_patients = sorted(p.name for p in VAD_ROOT.iterdir()
                          if p.is_dir() and p.name.startswith('Y'))
else:
    all_patients = list(PATIENTS)

overview = []
for pt in all_patients:
    mdays = get_model_days(pt)
    tdays = get_test_days(pt)
    n_pairs = sum(1 for (d, _, _) in mdays for t in tdays if t > d)
    overview.append(dict(patient=pt, n_model_days=len(mdays),
                         n_test_days=len(tdays), n_pairs=n_pairs))

overview_df = pd.DataFrame(overview)
print(f'Patients with model days: {int((overview_df.n_model_days > 0).sum())}')
print(f'Total (train, test) pairs: {int(overview_df.n_pairs.sum())}')
display(overview_df)

Patients with model days: 21
Total (train, test) pairs: 558


,patient,n_model_days,n_test_days,n_pairs
0,YEY,2,2,1
1,YEZ,6,6,15
2,YFA,9,9,36
3,YFB,9,9,36
4,YFC,9,9,36
5,YFD,6,6,15
6,YFE,4,4,6
7,YFF,10,10,45
8,YFG,5,5,10
9,YFI,7,7,21


## 2. Phase-1 Status and Submission

One job per (patient, train_date) — fits full-day GLM with fixed alpha.

In [8]:
def p1_paths(patient, train_date):
    drift_dir = VAD_ROOT / patient / 'functional_drift' / DRIFT_SUBDIR / train_date
    return dict(
        out_dir    = drift_dir,
        model_tar  = drift_dir / f'{patient}_fullday_model.tar',
        bundle_pkl = drift_dir / f'{patient}_fullday_bundle.pkl',
        train_idx  = drift_dir / f'{patient}_fullday_train_idx.npy',
        meta_json  = drift_dir / f'{patient}_fullday_meta.json',
        success    = drift_dir / f'{patient}_TRAIN_SUCCESS',
        error      = drift_dir / f'{patient}_error.txt',
    )


rows_p1 = []
for pt in all_patients:
    emb, cnt, dur = resolve_patient_inputs(pt)
    if any(p is None for p in [emb, cnt, dur]):
        continue
    for train_date, cv_pkl, word_idx_path in get_model_days(pt):
        pp = p1_paths(pt, train_date)
        done  = pp['success'].exists()
        error = pp['error'].exists()
        rows_p1.append(dict(
            patient=pt, train_date=train_date,
            cv_results_pkl=cv_pkl, word_idx_path=word_idx_path,
            embeddings_path=emb, counts_path=cnt, durations_path=dur,
            **pp,
            done=done, has_error=error,
            needs_p1=not done and (FORCE_RESUBMIT or not done),
        ))

p1_df = pd.DataFrame(rows_p1)
print(f'Phase-1 rows: {len(p1_df)}')
print(f'  done:        {p1_df.done.sum()}')
print(f'  errors:      {p1_df.has_error.sum()}')
print(f'  needs submit:{p1_df.needs_p1.sum()}')
p1_df[['patient','train_date','done','has_error','needs_p1']]

Phase-1 rows: 156
  done:        156
  errors:      0
  needs submit:0


,patient,train_date,done,has_error,needs_p1
0,YEY,2024-04-01,True,False,False
1,YEY,2024-04-02,True,False,False
2,YEZ,2024-04-09,True,False,False
3,YEZ,2024-04-10,True,False,False
4,YEZ,2024-04-11,True,False,False
...,...,...,...,...,...
151,YFU,2025-12-18,True,False,False
152,YFV,2026-02-17,True,False,False
153,YFV,2026-02-18,True,False,False
154,YFV,2026-02-19,True,False,False


In [9]:
DRY_RUN = False

submitted_p1 = {}   # (patient, train_date) -> job_id
failed_p1    = []

for _, row in p1_df[p1_df['needs_p1']].iterrows():
    pt         = row['patient']
    train_date = row['train_date']
    out_dir    = row['out_dir']
    out_dir.mkdir(parents=True, exist_ok=True)

    if FORCE_RESUBMIT:
        reset = f"rm -f {row['success']} {row['error']}"
    else:
        reset = 'true'

    gres_line = f'#SBATCH --gres={P1_GRES}' if P1_GRES else ''
    cmd = (
        f'{PYTHON_BIN} {WORKER_TRAIN} {pt} {VAD_ROOT} {out_dir} '
        f'--train-date {train_date} '
        f'--cv-results-pkl {row["cv_results_pkl"]} '
        f'--word-idx-path {row["word_idx_path"]} '
        f'--embeddings-path {row["embeddings_path"]} '
        f'--counts-path {row["counts_path"]} '
        f'--durations-path {row["durations_path"]} '
        f'--spike-offset-idx {SPIKE_OFFSET_IDX} '
        f'--gpt2-layer {GPT2_LAYER} '
        f'--n-pca {N_PCA}'
    )

    script = '\n'.join([
        '#!/bin/bash',
        f'#SBATCH --job-name=enc_drift_train_{pt}_{train_date}',
        f'#SBATCH --partition={P1_PARTITION}',
        gres_line,
        f'#SBATCH --cpus-per-task={P1_CPUS}',
        f'#SBATCH --qos=big_batch_tier',
        f'#SBATCH --exclude=guppy-1',
        f'#SBATCH --mem={P1_MEM}',
        f'#SBATCH --time={P1_TIME}',
        f'#SBATCH --output={LOGS_DIR}/{pt}_{train_date}_p1_%j.out',
        f'#SBATCH --error={LOGS_DIR}/{pt}_{train_date}_p1_%j.err',
        '',
        'set -eo pipefail',
        CONDA_ACTIVATE,
        'echo "HOST: $(hostname)  START: $(date)"',
        reset,
        cmd,
        'echo "END: $(date)"',
    ])

    sbatch_path = SCRIPTS_DIR / f'{pt}_{train_date}_p1.sbatch'
    sbatch_path.write_text(script)

    if DRY_RUN:
        print(f'[DRY] {pt} {train_date}')
        continue

    try:
        res = subprocess.run(['sbatch', '--parsable', str(sbatch_path)],
                             capture_output=True, text=True, check=True)
        jid = res.stdout.strip()
        submitted_p1[(pt, train_date)] = jid
        print(f'submitted P1: {pt} {train_date} -> {jid}')
    except subprocess.CalledProcessError as exc:
        failed_p1.append((pt, train_date, exc.stderr.strip()))
        print(f'FAILED P1: {pt} {train_date}\n{exc.stderr}')

print(f'\nP1 submitted={len(submitted_p1)}, failed={len(failed_p1)}')


P1 submitted=0, failed=0


## 3. Phase-2 Status and Submission

One job per (patient, train_date) — tests all subsequent days with permutation testing.
Depends on corresponding Phase-1 job.

In [11]:
def p2_success(patient, train_date, test_date, drift_dir):
    return drift_dir / f'{test_date}_DRIFT_SUCCESS'

def p2_result(patient, train_date, test_date, drift_dir):
    return drift_dir / f'{patient}_{train_date}_{test_date}_drift_results.pkl'


rows_p2 = []
for pt in all_patients:
    test_days = get_test_days(pt)
    for train_date, cv_pkl, word_idx_path in get_model_days(pt):
        pp = p1_paths(pt, train_date)
        if not pp['success'].exists():
            # Only submit P2 if P1 is done (or will be done via dependency)
            p1_pending = (pt, train_date) in submitted_p1
        else:
            p1_pending = False

        future_tests = [t for t in test_days if t > train_date]
        if not future_tests:
            continue

        drift_dir = pp['out_dir']
        all_test_done = all(
            p2_success(pt, train_date, td, drift_dir).exists()
            for td in future_tests
        )

        rows_p2.append(dict(
            patient=pt, train_date=train_date,
            drift_dir=drift_dir,
            future_tests=future_tests,
            p1_done=pp['success'].exists(),
            p1_pending=p1_pending,
            all_test_done=all_test_done,
            p1_jid=submitted_p1.get((pt, train_date), None),
        ))

p2_df = pd.DataFrame(rows_p2)
print(f'Phase-2 rows: {len(p2_df)}')
print(f'  P1 done:          {p2_df.p1_done.sum()}')
print(f'  P1 pending (submitted this run): {p2_df.p1_pending.sum()}')
print(f'  All tests done:   {p2_df.all_test_done.sum()}')
print(f'  Can submit P2:    {(p2_df.p1_done | p2_df.p1_pending).sum()}')

Phase-2 rows: 135
  P1 done:          135
  P1 pending (submitted this run): 0
  All tests done:   135
  Can submit P2:    135


In [16]:
DRY_RUN = False

submitted_p2 = {}
failed_p2    = []

for _, row in p2_df.iterrows():
    pt         = row['patient']
    train_date = row['train_date']

    # Skip if neither P1 done nor P1 submitted this run
    if not row['p1_done'] and not row['p1_pending']:
        continue

    # Skip if all test dates already processed
    if row['all_test_done'] and not FORCE_RESUBMIT:
        continue

    drift_dir = row['drift_dir']
    drift_dir.mkdir(parents=True, exist_ok=True)

    dep_flag = ''
    if row['p1_jid']:
        dep_flag = f'--dependency=afterok:{row["p1_jid"]}'
    elif not row['p1_done']:
        print(f'  skipping {pt} {train_date}: P1 not done and no pending job ID')
        continue

    gres_line = f'#SBATCH --gres={P2_GRES}' if P2_GRES else ''
    cmd = (
        f'{PYTHON_BIN} {WORKER_TEST} {pt} {VAD_ROOT} '
        f'--train-date {train_date} '
        f'--train-out-dir {drift_dir} '
        f'--source-run {SOURCE_RUN} '
        f'--n-permutations {N_PERMUTATIONS} '
        f'--n-jobs {N_JOBS_PERM} '
        f'--spike-offset-idx {SPIKE_OFFSET_IDX} '
        f'--gpt2-layer {GPT2_LAYER}'
    )

    script = '\n'.join([
        '#!/bin/bash',
        f'#SBATCH --job-name=enc_drift_test_{pt}_{train_date}',
        f'#SBATCH --partition={P2_PARTITION}',
        gres_line,
        f'#SBATCH --cpus-per-task={P2_CPUS}',
        f'#SBATCH --qos=big_batch_tier',
        f'#SBATCH --exclude=guppy-1',
        f'#SBATCH --mem={P2_MEM}',
        f'#SBATCH --time={P2_TIME}',
        f'#SBATCH --output={LOGS_DIR}/{pt}_{train_date}_p2_%j.out',
        f'#SBATCH --error={LOGS_DIR}/{pt}_{train_date}_p2_%j.err',
        '',
        'set -eo pipefail',
        CONDA_ACTIVATE,
        'echo "HOST: $(hostname)  START: $(date)"',
        cmd,
        'echo "END: $(date)"',
    ])

    sbatch_path = SCRIPTS_DIR / f'{pt}_{train_date}_p2.sbatch'
    sbatch_path.write_text(script)

    if DRY_RUN:
        print(f'[DRY] P2: {pt} {train_date} dep={dep_flag}')
        continue

    try:
        sbatch_args = ['sbatch', '--parsable']
        if dep_flag:
            sbatch_args.append(dep_flag)
        sbatch_args.append(str(sbatch_path))
        res = subprocess.run(sbatch_args, capture_output=True, text=True, check=True)
        jid = res.stdout.strip()
        submitted_p2[(pt, train_date)] = jid
        print(f'submitted P2: {pt} {train_date} -> {jid}  dep={dep_flag or "none"}')
    except subprocess.CalledProcessError as exc:
        failed_p2.append((pt, train_date, exc.stderr.strip()))
        print(f'FAILED P2: {pt} {train_date}\n{exc.stderr}')

print(f'\nP2 submitted={len(submitted_p2)}, failed={len(failed_p2)}')

submitted P2: YEY 2024-04-01 -> 483089  dep=none
submitted P2: YEZ 2024-04-09 -> 483090  dep=none
submitted P2: YEZ 2024-04-10 -> 483091  dep=none
submitted P2: YEZ 2024-04-11 -> 483092  dep=none
submitted P2: YEZ 2024-04-12 -> 483093  dep=none
submitted P2: YEZ 2024-04-14 -> 483094  dep=none
submitted P2: YFA 2024-04-23 -> 483095  dep=none
submitted P2: YFA 2024-04-24 -> 483096  dep=none
submitted P2: YFA 2024-04-25 -> 483097  dep=none
submitted P2: YFA 2024-04-26 -> 483098  dep=none
submitted P2: YFA 2024-04-27 -> 483099  dep=none
submitted P2: YFA 2024-04-28 -> 483100  dep=none
submitted P2: YFA 2024-04-29 -> 483101  dep=none
submitted P2: YFA 2024-04-30 -> 483102  dep=none
submitted P2: YFB 2024-05-02 -> 483103  dep=none
submitted P2: YFB 2024-05-03 -> 483104  dep=none
submitted P2: YFB 2024-05-04 -> 483105  dep=none
submitted P2: YFB 2024-05-05 -> 483106  dep=none
submitted P2: YFB 2024-05-06 -> 483107  dep=none
submitted P2: YFB 2024-05-07 -> 483108  dep=none
submitted P2: YFB 20

## 4. Check Status

In [13]:
rows_status = []
for pt in all_patients:
    test_days = get_test_days(pt)
    for train_date, cv_pkl, _ in get_model_days(pt):
        pp = p1_paths(pt, train_date)
        drift_dir = pp['out_dir']
        future_tests = [t for t in test_days if t > train_date]
        for test_date in future_tests:
            rows_status.append(dict(
                patient=pt, train_date=train_date, test_date=test_date,
                p1_done=pp['success'].exists(),
                p2_done=p2_success(pt, train_date, test_date, drift_dir).exists(),
                result_exists=p2_result(pt, train_date, test_date, drift_dir).exists(),
                p2_error=(drift_dir / f'{test_date}_error.txt').exists(),
            ))

status_df = pd.DataFrame(rows_status)
print(f'Total (train, test) pairs: {len(status_df)}')
print(f'  P1 done:     {status_df.p1_done.sum()}')
print(f'  P2 done:     {status_df.p2_done.sum()}')
print(f'  P2 errors:   {status_df.p2_error.sum()}')
status_df[['patient','train_date','test_date','p1_done','p2_done','result_exists','p2_error']]

Total (train, test) pairs: 558
  P1 done:     558
  P2 done:     558
  P2 errors:   558


,patient,train_date,test_date,p1_done,p2_done,result_exists,p2_error
0,YEY,2024-04-01,2024-04-02,True,True,True,True
1,YEZ,2024-04-09,2024-04-10,True,True,True,True
2,YEZ,2024-04-09,2024-04-11,True,True,True,True
3,YEZ,2024-04-09,2024-04-12,True,True,True,True
4,YEZ,2024-04-09,2024-04-14,True,True,True,True
...,...,...,...,...,...,...,...
553,YFV,2026-02-17,2026-02-19,True,True,True,True
554,YFV,2026-02-17,2026-02-20,True,True,True,True
555,YFV,2026-02-18,2026-02-19,True,True,True,True
556,YFV,2026-02-18,2026-02-20,True,True,True,True


## 5. Assemble Results

In [14]:
all_results = []

for pt in all_patients:
    test_days = get_test_days(pt)
    for train_date, cv_pkl, _ in get_model_days(pt):
        drift_dir = VAD_ROOT / pt / 'functional_drift' / DRIFT_SUBDIR / train_date
        for test_date in [t for t in test_days if t > train_date]:
            result_path = p2_result(pt, train_date, test_date, drift_dir)
            if not result_path.exists():
                continue
            with open(result_path, 'rb') as f:
                res = pickle.load(f)
            # Flatten per-neuron metrics into rows
            K = res['n_neurons']
            for n in range(K):
                all_results.append(dict(
                    patient=res['patient'],
                    train_date=res['train_date'],
                    test_date=res['test_date'],
                    day_offset=res['day_offset'],
                    neuron_idx=n,
                    n_train_words=res['n_train_words'],
                    n_test_words=res['n_test_words'],
                    n_permutations=res['n_permutations'],
                    ll_real=float(res['ll_real'][n]),
                    ll_null=float(res['ll_null'][n]),
                    pseudo_r2=float(res['pseudo_r2'][n]),
                    pearson_corr=float(res['pearson_corr'][n]),
                    spearman_corr=float(res['spearman_corr'][n]),
                    ll_perm_mean=float(res['ll_perm_mean'][n]),
                    p_val_ll=float(res['p_val_ll'][n]),
                ))

if all_results:
    combined = pd.DataFrame(all_results)
    out_path = VAD_ROOT / f'functional_drift_encoding_{SPEECH_TYPE}_results_all.pkl'
    with open(out_path, 'wb') as f:
        pickle.dump(combined, f)
    print(f'combined: {len(combined)} rows -> {out_path}')
    display(combined.head())
else:
    print('No results assembled yet.')

combined: 30376 rows -> /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/functional_drift_encoding_filtered_results_all.pkl


,patient,train_date,test_date,day_offset,neuron_idx,n_train_words,n_test_words,n_permutations,ll_real,ll_null,pseudo_r2,pearson_corr,spearman_corr,ll_perm_mean,p_val_ll
0,YEY,2024-04-01,2024-04-02,1,0,16510,23841,50,-141534.478992,-152170.478845,0.069895,0.342066,0.335398,-104139.522091,1.0
1,YEY,2024-04-01,2024-04-02,1,1,16510,23841,50,-147285.178273,-162704.733553,0.094770,0.340245,0.342875,-111998.854695,1.0
2,YEY,2024-04-01,2024-04-02,1,2,16510,23841,50,-129569.105862,-145176.722839,0.107508,0.380172,0.377938,-95171.134402,1.0
3,YEY,2024-04-01,2024-04-02,1,3,16510,23841,50,-138940.213881,-154718.632037,0.101981,0.356926,0.349688,-104876.918615,1.0
4,YEY,2024-04-01,2024-04-02,1,4,16510,23841,50,-169459.024473,-179719.465229,0.057091,0.299465,0.314340,-128160.253381,1.0


## 6. Summary Statistics

In [15]:
if all_results:
    combined = pd.DataFrame(all_results) if not isinstance(all_results, pd.DataFrame) else combined

    # Average across neurons per (patient, train_date, test_date)
    pair_avg = (
        combined.groupby(['patient','train_date','test_date','day_offset'])
        .agg(
            pseudo_r2_mean=('pseudo_r2','mean'),
            pseudo_r2_std=('pseudo_r2','std'),
            pearson_mean=('pearson_corr','mean'),
            spearman_mean=('spearman_corr','mean'),
            p_val_ll_mean=('p_val_ll','mean'),
            frac_sig_05=('p_val_ll', lambda x: (x < 0.05).mean()),
            n_neurons=('neuron_idx','nunique'),
        )
        .reset_index()
    )

    by_offset = (
        pair_avg.groupby('day_offset')
        .agg(
            n_pairs=('patient','count'),
            pseudo_r2_mean=('pseudo_r2_mean','mean'),
            pseudo_r2_std=('pseudo_r2_mean','std'),
            pearson_mean=('pearson_mean','mean'),
            frac_sig_05_mean=('frac_sig_05','mean'),
        )
        .reset_index()
    )
    print('By day offset:')
    display(by_offset)

By day offset:


,day_offset,n_pairs,pseudo_r2_mean,pseudo_r2_std,pearson_mean,frac_sig_05_mean
0,1,133,0.238725,0.090503,0.529363,0.080371
1,2,112,0.231519,0.078991,0.527042,0.103230
2,3,93,0.212280,0.085050,0.511422,0.097539
3,4,74,0.193230,0.110359,0.503861,0.087822
4,5,57,0.186095,0.105966,0.489543,0.106310
5,6,40,0.179422,0.079564,0.485191,0.090420
6,7,26,0.142584,0.151636,0.442855,0.088782
7,8,15,0.138314,0.133732,0.448270,0.096081
8,9,6,0.180354,0.066294,0.503970,0.044643
9,10,2,0.205137,0.008993,0.511291,0.031250
